# B1 — Uber Fare Prediction: Linear vs Random Forest, outliers, correlation, R2, RMSE

**Laboratory Practice III (410246), Group B**

Dataset reference: https://www.kaggle.com/yasserh/uber-fares-dataset

Run cells top-to-bottom. Set `CSV_PATH` to your downloaded CSV file to use real data; `None` runs the included synthetic demonstration. Results on synthetic data are not real-world model performance.

In [ ]:
# Set to a local CSV filename/path after downloading the Kaggle dataset.
# Keep None to run with built-in synthetic demonstration data.
CSV_PATH = None  # Example: 'uber.csv'

## Implementation and evaluation

In [ ]:
"""B1: Uber fare regression, outliers, correlations, R2 and RMSE."""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

def main():
    if CSV_PATH:  # Load actual Uber CSV here: --csv path/to/uber.csv
        df=pd.read_csv(CSV_PATH)
    else:
        rng=np.random.default_rng(42);n=400
        distance=rng.uniform(0.5,25,n)
        df=pd.DataFrame({'distance_km':distance,'passenger_count':rng.integers(1,5,n),
                         'fare_amount':4+2.2*distance+rng.normal(0,2,n)})
        print('Using synthetic demonstration data; provide --csv for actual data.')
    if 'fare_amount' not in df: raise ValueError('CSV must contain fare_amount')
    # Derive trip distance from Uber pickup/dropoff coordinates when present.
    coords=['pickup_latitude','pickup_longitude','dropoff_latitude','dropoff_longitude']
    if all(c in df for c in coords):
        lat1=np.radians(pd.to_numeric(df[coords[0]],errors='coerce'))
        lon1=np.radians(pd.to_numeric(df[coords[1]],errors='coerce'))
        lat2=np.radians(pd.to_numeric(df[coords[2]],errors='coerce'))
        lon2=np.radians(pd.to_numeric(df[coords[3]],errors='coerce'))
        a=np.sin((lat2-lat1)/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
        df['distance_km']=6371*2*np.arcsin(np.sqrt(np.clip(a,0,1)))
    if 'pickup_datetime' in df:
        times=pd.to_datetime(df['pickup_datetime'],errors='coerce',utc=True)
        df['pickup_hour']=times.dt.hour
        df['pickup_weekday']=times.dt.dayofweek
    df['fare_amount']=pd.to_numeric(df['fare_amount'],errors='coerce')
    df=df.replace([np.inf,-np.inf],np.nan).dropna(subset=['fare_amount'])
    df=df[df.fare_amount>0].copy()
    # IQR-based fare outlier detection (filter only after splitting to avoid leakage).
    if len(df)<15: raise ValueError('At least 15 valid rows required')
    excluded={'fare_amount','key','Unnamed: 0','pickup_datetime','pickup_latitude','pickup_longitude','dropoff_latitude','dropoff_longitude'}
    X=df.drop(columns=[c for c in excluded if c in df]).copy()
    X=X.select_dtypes(include=['number','object','category','bool'])
    X=X.dropna(axis=1,how='all')
    if X.empty: raise ValueError('No usable predictor columns')
    y=df['fare_amount']
    print('Correlation with fare:\n',df.select_dtypes(include='number').corr()['fare_amount'].sort_values(ascending=False).head(12))
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
    q1,q3=y_train.quantile([.25,.75]);iqr=q3-q1
    mask=y_train.between(q1-1.5*iqr,q3+1.5*iqr)
    print('Training fare outliers:',int((~mask).sum()))
    X_train,y_train=X_train.loc[mask],y_train.loc[mask]
    numeric=X.select_dtypes(include='number').columns.tolist()
    categorical=[c for c in X if c not in numeric]
    transformers=[]
    if numeric:transformers.append(('num',SimpleImputer(strategy='median'),numeric))
    if categorical:transformers.append(('cat',Pipeline([('fill',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical))
    for name,model in [('Linear Regression',LinearRegression()),('Random Forest',RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1))]:
        pipe=Pipeline([('prep',ColumnTransformer(transformers)),('model',model)])
        pipe.fit(X_train,y_train);pred=pipe.predict(X_test)
        print(f'{name}: R2={r2_score(y_test,pred):.4f}, RMSE={np.sqrt(mean_squared_error(y_test,pred)):.4f}')
main()
